In [1]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from branca.colormap import linear
import networkx as nx
import matplotlib.pyplot as plt
from geopy.distance import geodesic
import itertools

In [3]:
df2 = pd.read_csv(
    "/Users/choejeonghun/Downloads/2024_bike/서울특별시 공공자전거 대여이력 정보_2024/서울특별시 공공자전거 대여이력 정보_2412.csv",
    encoding="cp949",
    low_memory=False
)


In [4]:
df2.head()

,자전거번호,대여일시,대여 대여소번호,대여 대여소명,대여거치대,반납일시,반납대여소번호,반납대여소명,반납거치대,이용시간(분),이용거리(M),생년,성별,이용자종류,대여대여소ID,반납대여소ID,자전거구분
0,SPB-38138,2024-12-01 00:00:13,228,선유도역 3번출구 앞,0,2024-12-01 00:02:41,04560,양평동성원아파트,0,2,625.10,1997,M,내국인,ST-278,ST-2811,일반자전거
1,SPB-54015,2024-12-01 00:01:22,1192,마곡수명산파크 209동 건너편,0,2024-12-01 00:03:31,02732,마곡수명산 1-2단지,0,2,0.00,1983,NaN,내국인,ST-1710,ST-2049,일반자전거
2,SPB-59421,2024-12-01 00:00:24,245,삼성생명 당산사옥 앞,0,2024-12-01 00:04:11,00280,양평동6차현대아파트 앞,0,3,890.00,1991,\N,내국인,ST-294,ST-1539,일반자전거
3,SPB-32929,2024-12-01 00:00:34,1117,등촌5단지아파트 버스정류장,0,2024-12-01 00:04:50,01174,강서구청사거리(부민병원),0,4,671.61,1994,\N,내국인,ST-834,ST-1511,일반자전거
4,SPB-39202,2024-12-01 00:01:15,1264,천호역 10번 출구 앞,0,2024-12-01 00:04:54,02611,송파지역자활센터 뒤,0,3,49.16,\N,M,내국인,ST-1083,ST-1684,일반자전거


In [7]:
Daily = pd.read_csv(
    "/Users/choejeonghun/Downloads/2024_bike/서울특별시 공공자전거 일별 대여건수_24.7-12.csv",
    encoding="cp949",
    low_memory=False
)


In [9]:
Daily.head()

,대여일자,대여건수
0,2024-07-01,182412
1,2024-07-02,39425
2,2024-07-03,164923
3,2024-07-04,158343
4,2024-07-05,170995


In [11]:
Time = pd.read_csv(
    "/Users/choejeonghun/Downloads/2024_bike/서울특별시 공공자전거 이용정보(시간대별)_202412.csv",
    encoding="cp949",
    low_memory=False
)


In [12]:
Time.head()

,대여일자,대여시간,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
0,2024-12-01,0,1442,1442. 중랑구청 중화동 별관 앞,정기권,NaN,~10대,1,105.32,0.66,2829.30,18
1,2024-12-01,0,2728,2728.마곡나루역 3번 출구,정기권,NaN,~10대,1,22.45,0.25,1090.00,7
2,2024-12-01,0,1023,1023. 한국종합기술사옥 앞,정기권,NaN,20대,1,148.31,0.87,3745.23,21
3,2024-12-01,0,1150,1150. 송정역 1번출구,정기권,NaN,20대,1,24.01,0.19,808.44,6
4,2024-12-01,0,1260,1260. 방이동 한양3차아파트 옆,정기권,NaN,20대,1,92.85,0.94,4042.55,35


In [15]:
Month = pd.read_csv(
    "/Users/choejeonghun/Downloads/2024_bike/서울특별시 공공자전거 이용정보(월별)_24.7-12.csv",
    encoding="cp949",
    low_memory=False
)


In [16]:
Month.head()

,대여일자,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
0,202407,102,102. 망원역 1번출구 앞,일일권,NaN,20대,67,4304.28,40.87,176252.62,1895
1,202407,102,102. 망원역 1번출구 앞,일일권,NaN,30대,64,4142.09,38.92,167774.15,1498
2,202407,102,102. 망원역 1번출구 앞,일일권,NaN,40대,2,265.91,2.11,9098.64,64
3,202407,102,102. 망원역 1번출구 앞,일일권,NaN,50대,6,648.52,5.11,22048.26,149
4,202407,102,102. 망원역 1번출구 앞,일일권,NaN,60대,1,35.37,0.31,1333.18,6


In [17]:
Forienr = pd.read_csv(
    '/Users/choejeonghun/Downloads/2024_bike/서울특별시 공공자전거 외국인 대여정보(일별)_24.7-12.csv', 
    encoding='cp949', 
    low_memory=False
)


In [18]:
Forienr.head()

,일시,대여소명,대여건수,반납건수
0,2024-07-01,108. 서교동 사거리,1,0
1,2024-07-01,505. 자양사거리 광진아크로텔 앞,0,1
2,2024-07-01,"1153. 발산역 1번, 9번 인근 대여소",1,0
3,2024-07-01,510. 뚝도아리수정수센터 버스정류소 옆,3,2
4,2024-07-01,511. 서울숲역 5번 출구 옆,1,3


In [23]:
Broken = pd.read_csv(
    '/Users/choejeonghun/Downloads/2024_bike/서울시 공공자전거 고장신고 내역_2407-2412.csv', 
    encoding='cp949', 
    low_memory=False
)


In [25]:
Broken.head()

,자전거번호,등록일시,구분
0,SPB-54796,2024-07-01 00:03:39,기타
1,SPB-65834,2024-07-01 00:12:26,기타
2,SPB-44251,2024-07-01 00:17:06,단말기
3,SPB-44251,2024-07-01 00:17:06,기타
4,SPB-53458,2024-07-01 00:31:19,페달


In [27]:
count  =pd.read_csv(
    '/Users/choejeonghun/Downloads/2024_bike/data_2412.csv', 
    encoding='cp949', 
    low_memory=False
)


In [28]:
count.head()

,일시,대여소번호,대여소명,시간대,거치대수량
0,2024-12-01,101,101. (구)합정동 주민센터,0,0
1,2024-12-01,102,102. 망원역 1번출구 앞,0,36
2,2024-12-01,103,103. 망원역 2번출구 앞,0,15
3,2024-12-01,104,104. 합정역 1번출구 앞,0,1
4,2024-12-01,105,105. 합정역 5번출구 앞,0,1


In [29]:
count.tail()

,일시,대여소번호,대여소명,시간대,거치대수량
2335244,2024-12-31,6172,6172. 가양5단지아파트,23,6
2335245,2024-12-31,6173,6173. 서울자동차운전전문학원,23,10
2335246,2024-12-31,6176,6176. 유광사 여성병원 앞,23,2
2335247,2024-12-31,6177,6177. 마곡롯데캐슬르웨스트,23,12
2335248,2024-12-31,6178,6178. 마스터밸류에이스 지식산업센터,23,12


In [30]:
New_Month = pd.read_csv(
    '/Users/choejeonghun/Downloads/2024_bike/서울특별시 공공자전거 신규가입자 정보(월별)_24.7-12.csv', 
    encoding='cp949', 
    low_memory=False
)

In [31]:
New_Month.head()

,가입년월,회원구분,연령대,성별,가입건수
0,202407,회원-내국인,~10대,F,2622
1,202407,회원-내국인,20대,F,4544
2,202407,회원-내국인,30대,F,2253
3,202407,회원-내국인,40대,F,1846
4,202407,회원-내국인,50대,F,1083


In [32]:
New_Daily = pd.read_csv(
    '/Users/choejeonghun/Downloads/2024_bike/서울특별시 공공자전거 신규가입자 정보(일별)_24.7-12.csv', 
    encoding='cp949', 
    low_memory=False
)

In [33]:
New_Daily.head()

,2024-07-01,회원-내국인,10대,F,77
0,2024-07-01,회원-내국인,10대,M,151
1,2024-07-01,회원-내국인,20대,F,262
2,2024-07-01,회원-내국인,20대,M,329
3,2024-07-01,회원-내국인,30대,F,118
4,2024-07-01,회원-내국인,30대,M,176


In [34]:
record_1201 = pd.read_csv(
    '/Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241201.csv', 
    encoding='cp949', 
    low_memory=False
)

In [42]:
record_1201.head()

,기준_날짜,집계_기준,기준_시간대,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,전체_이용_분,전체_이용_거리
0,20241201,도착시간,1515,ST-3182,구로2동_001_6,ST-1976,문래동_032_1,1,16.0,2204.0
1,20241201,도착시간,1200,ST-547,잠실2동_058_1,ST-551,잠실3동_029_1,1,5.0,752.0
2,20241201,도착시간,1125,ST-1065,가양1동_016_2,ST-1688,가양1동_016_3,1,3.0,548.0
3,20241201,도착시간,1320,ST-2749,길동_012_1,ST-2749,길동_012_1,1,97.0,1216.0
4,20241201,도착시간,100,ST-3238,영등포동_056_1,ST-299,신길7동_009_1,1,18.0,2828.0


In [45]:



december_list = []


for day in range(1, 32):

    # 날짜 포맷
    day_str = f"{day:02d}"

    # 파일 경로
    file_path = (
        f"/Users/choejeonghun/Downloads/2024_bike/"
        f"tpss_bcycl_od_statnhm_202412/"
        f"tpss_bcycl_od_statnhm_202412{day_str}.csv"
    )

    print(f"Loading: {file_path}")

    
    temp_df = pd.read_csv(
        file_path,
        encoding='cp949',
        low_memory=False
    )

   
    temp_df['record_date'] = f"2024-12-{day_str}"


    december_list.append(temp_df)


df_december = pd.concat(
    december_list,
    ignore_index=True
)


print(df_december.head())

print(df_december.shape)

print(df_december.columns)


df_december.to_csv(
    "/Users/choejeonghun/Downloads/2024_bike/december_2024_merged.csv",
    index=False,
    encoding='utf-8-sig'
)

Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241201.csv
Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241202.csv
Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241203.csv
Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241204.csv
Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241205.csv
Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241206.csv
Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241207.csv
Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_statnhm_20241208.csv
Loading: /Users/choejeonghun/Downloads/2024_bike/tpss_bcycl_od_statnhm_202412/tpss_bcycl_od_stat

In [47]:




df_december = pd.read_csv(
    "/Users/choejeonghun/Downloads/2024_bike/december_2024_merged.csv",
    encoding='utf-8-sig',
    low_memory=False
)


print(df_december.head())

print(df_december.shape)

print(df_december.columns)

      기준_날짜 집계_기준  기준_시간대 시작_대여소_ID     시작_대여소명 종료_대여소_ID     종료_대여소명  전체_건수  \
0  20241201  도착시간    1515   ST-3182  구로2동_001_6   ST-1976   문래동_032_1      1   
1  20241201  도착시간    1200    ST-547  잠실2동_058_1    ST-551  잠실3동_029_1      1   
2  20241201  도착시간    1125   ST-1065  가양1동_016_2   ST-1688  가양1동_016_3      1   
3  20241201  도착시간    1320   ST-2749    길동_012_1   ST-2749    길동_012_1      1   
4  20241201  도착시간     100   ST-3238  영등포동_056_1    ST-299  신길7동_009_1      1   

   전체_이용_분  전체_이용_거리 record_date  
0     16.0    2204.0  2024-12-01  
1      5.0     752.0  2024-12-01  
2      3.0     548.0  2024-12-01  
3     97.0    1216.0  2024-12-01  
4     18.0    2828.0  2024-12-01  
(4239494, 11)
Index(['기준_날짜', '집계_기준', '기준_시간대', '시작_대여소_ID', '시작_대여소명', '종료_대여소_ID',
       '종료_대여소명', '전체_건수', '전체_이용_분', '전체_이용_거리', 'record_date'],
      dtype='object')


In [48]:
# Remove whitespace from column names
df2.columns = df2.columns.str.strip()

# Rename columns to English
df2 = df2.rename(columns={

    '자전거번호': 'bike_number',
    '자전거 번호': 'bike_number',

    '대여일시': 'rental_datetime',
    '반납일시': 'return_datetime',

    '대여 대여소번호': 'rental_station_id',
    '대여 대여소명': 'rental_station_name',

    '반납대여소번호': 'return_station_id',
    '반납대여소명': 'return_station_name',

    '이용시간(분)': 'usage_time',
    '이용거리(M)': 'usage_distance',

    '성별': 'gender',
    '연령대코드': 'age_group',

    '생년': 'birth_year',
    '이용건수': 'usage_count',
    
    '반납거치대': 'return_rack',
    '이용자종류': 'user_type' ,

    '자전거구분': 'bike_type',
    '이용자종류': 'user_type'
    
})

# Check columns
print(df2.columns)

# Preview data
print(df2.head())

Index(['bike_number', 'rental_datetime', 'rental_station_id',
       'rental_station_name', '대여거치대', 'return_datetime', 'return_station_id',
       'return_station_name', 'return_rack', 'usage_time', 'usage_distance',
       'birth_year', 'gender', 'user_type', '대여대여소ID', '반납대여소ID', 'bike_type'],
      dtype='object')
  bike_number      rental_datetime  rental_station_id rental_station_name  \
0   SPB-38138  2024-12-01 00:00:13                228         선유도역 3번출구 앞   
1   SPB-54015  2024-12-01 00:01:22               1192    마곡수명산파크 209동 건너편   
2   SPB-59421  2024-12-01 00:00:24                245         삼성생명 당산사옥 앞   
3   SPB-32929  2024-12-01 00:00:34               1117      등촌5단지아파트 버스정류장   
4   SPB-39202  2024-12-01 00:01:15               1264        천호역 10번 출구 앞   

   대여거치대      return_datetime return_station_id return_station_name  \
0      0  2024-12-01 00:02:41             04560            양평동성원아파트   
1      0  2024-12-01 00:03:31             02732         마곡수명산 1-2단지   
2   

In [49]:

station_master = pd.read_excel(

    "/Users/choejeonghun/Downloads/2024_bike/공공자전거 대여소 정보(24.12월 기준).xlsx",

    sheet_name='대여소현황',

    header=4

)


print(station_master.head())

print(station_master.columns)

   Unnamed: 0    Unnamed: 1 Unnamed: 2                       Unnamed: 3  \
0         301   경복궁역 7번출구 앞        종로구  서울특별시 종로구 사직로 지하130 경복궁역 7번출구 앞   
1         302   경복궁역 4번출구 뒤        종로구  서울특별시 종로구 사직로 지하130 경복궁역 4번출구 뒤   
2         303   광화문역 1번출구 앞        종로구       서울특별시 종로구 세종대로 지하189 세종로공원   
3         305        종로구청 옆        종로구               서울특별시 종로구 세종로 84-1   
4         307     서울역사박물관 앞        종로구      서울특별시 종로구 새문안로 55 서울역사박물관 앞   

   Unnamed: 4  Unnamed: 5          Unnamed: 6  Unnamed: 7  Unnamed: 8  \
0   37.575794  126.971451 2015-10-07 12:03:46         NaN        20.0   
1   37.575947  126.974060 2015-10-07 12:04:22         NaN        12.0   
2   37.571770  126.974663 2015-10-07 00:00:00         NaN         8.0   
3   37.572559  126.978333 2015-01-07 00:00:00         NaN        16.0   
4   37.570000  126.971100 2015-10-07 12:09:09         NaN        11.0   

  Unnamed: 9  
0         QR  
1         QR  
2         QR  
3         QR  
4         QR  
Index(['Unnamed: 0',

In [50]:
station_master = station_master.rename(columns={

    '대여소\n번호': 'station_id',

    '보관소(대여소)명': 'station_name',

    '소재지(위치)': 'district',

    'Unnamed: 3': 'address',

    'Unnamed: 4': 'latitude',

    'Unnamed: 5': 'longitude',

    '설치형태': 'lcd_rack_count',

    'Unnamed: 8': 'qr_rack_count',

    '운영\n방식': 'operation_type'

})

In [51]:
station_master.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,address,latitude,longitude,Unnamed: 6,Unnamed: 7,qr_rack_count,Unnamed: 9
0,301,경복궁역 7번출구 앞,종로구,서울특별시 종로구 사직로 지하130 경복궁역 7번출구 앞,37.575794,126.971451,2015-10-07 12:03:46,NaN,20.0,QR
1,302,경복궁역 4번출구 뒤,종로구,서울특별시 종로구 사직로 지하130 경복궁역 4번출구 뒤,37.575947,126.974060,2015-10-07 12:04:22,NaN,12.0,QR
2,303,광화문역 1번출구 앞,종로구,서울특별시 종로구 세종대로 지하189 세종로공원,37.571770,126.974663,2015-10-07 00:00:00,NaN,8.0,QR
3,305,종로구청 옆,종로구,서울특별시 종로구 세종로 84-1,37.572559,126.978333,2015-01-07 00:00:00,NaN,16.0,QR
4,307,서울역사박물관 앞,종로구,서울특별시 종로구 새문안로 55 서울역사박물관 앞,37.570000,126.971100,2015-10-07 12:09:09,NaN,11.0,QR


In [52]:
station_master = station_master.rename(columns={

    'Unnamed: 0': 'station_id',

    'Unnamed: 1': 'station_name',

    'Unnamed: 2': 'district',

    'Unnamed: 6': 'installation_date',

    'Unnamed: 7': 'lcd_rack_count',

    'qr_rack_count': 'qr_rack_count',

    'Unnamed: 9': 'operation_type'

})

In [59]:
station_master['station_id'] = (

    station_master['station_id']

    .astype(str)

    .str.replace('.0', '', regex=False)

    .str.strip()

)

In [61]:
print(station_master.columns)

station_master.head()

Index(['station_id', 'station_name', 'district', 'address', 'latitude',
       'longitude', 'installation_date', 'lcd_rack_count', 'qr_rack_count',
       'operation_type'],
      dtype='object')


,station_id,station_name,district,address,latitude,longitude,installation_date,lcd_rack_count,qr_rack_count,operation_type
0,301,경복궁역 7번출구 앞,종로구,서울특별시 종로구 사직로 지하130 경복궁역 7번출구 앞,37.575794,126.971451,2015-10-07 12:03:46,NaN,20.0,QR
1,302,경복궁역 4번출구 뒤,종로구,서울특별시 종로구 사직로 지하130 경복궁역 4번출구 뒤,37.575947,126.974060,2015-10-07 12:04:22,NaN,12.0,QR
2,303,광화문역 1번출구 앞,종로구,서울특별시 종로구 세종대로 지하189 세종로공원,37.571770,126.974663,2015-10-07 00:00:00,NaN,8.0,QR
3,305,종로구청 옆,종로구,서울특별시 종로구 세종로 84-1,37.572559,126.978333,2015-01-07 00:00:00,NaN,16.0,QR
4,307,서울역사박물관 앞,종로구,서울특별시 종로구 새문안로 55 서울역사박물관 앞,37.570000,126.971100,2015-10-07 12:09:09,NaN,11.0,QR


In [63]:
station_master['operation_type'].value_counts()

operation_type
QR        1682
LCD       1078
LCD,QR       6
Name: count, dtype: int64

In [65]:
station_master['total_rack_count'] = (

    station_master['lcd_rack_count']

    .fillna(0)

    +

    station_master['qr_rack_count']

    .fillna(0)

)

In [67]:
station_master['qr_ratio'] = (

    station_master['qr_rack_count']

    /

    station_master['total_rack_count']

)

In [69]:
station_master['installation_date'] = pd.to_datetime(

    station_master['installation_date']

)

In [71]:
reference_date = pd.Timestamp('2026-05-26')

In [73]:
station_master['station_age_days'] = (

    reference_date

    -

    station_master['installation_date']

).dt.days

In [75]:
station_master[[

    'station_id',
    'station_name',
    'installation_date',
    'station_age_days'

]].head()

,station_id,station_name,installation_date,station_age_days
0,301,경복궁역 7번출구 앞,2015-10-07 12:03:46,3883
1,302,경복궁역 4번출구 뒤,2015-10-07 12:04:22,3883
2,303,광화문역 1번출구 앞,2015-10-07 00:00:00,3884
3,305,종로구청 옆,2015-01-07 00:00:00,4157
4,307,서울역사박물관 앞,2015-10-07 12:09:09,3883


In [77]:
df2['station_id'] = (

    df2['rental_station_id']

    .astype(str)

    .str.replace('.0', '', regex=False)

    .str.strip()

)

In [78]:
avg_trip_duration = (

    df2.groupby('station_id')['usage_time']

    .mean()

    .rename('avg_trip_duration')

    .reset_index()

)

In [81]:
station_master = station_master.merge(

    avg_trip_duration,

    on='station_id',

    how='left'

)

In [83]:
station_master[[

    'station_id',
    'station_name',
    'avg_trip_duration'

]].head()

,station_id,station_name,avg_trip_duration
0,301,경복궁역 7번출구 앞,15.625478
1,302,경복궁역 4번출구 뒤,16.966642
2,303,광화문역 1번출구 앞,15.733066
3,305,종로구청 옆,21.576068
4,307,서울역사박물관 앞,20.488591


In [85]:
df2['distance_km'] = (
    df2['usage_distance'] / 1000
)

In [87]:
avg_distance = (

    df2.groupby('station_id')['distance_km']

    .mean()

    .rename('avg_distance')

    .reset_index()

)

In [89]:
station_master = station_master.merge(

    avg_distance,

    on='station_id',

    how='left'

)

In [91]:
df2['is_round_trip'] = (

    df2['rental_station_id']

    ==

    df2['return_station_id']

).astype(int)

In [93]:
round_trip_ratio = (

    df2.groupby('station_id')['is_round_trip']

    .mean()

    .rename('round_trip_ratio')

    .reset_index()

)

In [95]:
station_master = station_master.merge(

    round_trip_ratio,

    on='station_id',

    how='left'

)

In [97]:
df2['bike_type'].value_counts()

bike_type
일반자전거    2252800
새싹자전거      25119
Name: count, dtype: int64

In [99]:
df2['is_sprout_bike'] = (

    df2['bike_type']

    .astype(str)

    .str.contains('새싹')

).astype(int)

In [101]:
sprout_bike_ratio = (

    df2.groupby('station_id')['is_sprout_bike']

    .mean()

    .rename('sprout_bike_ratio')

    .reset_index()

)

In [103]:
station_master = station_master.merge(

    sprout_bike_ratio,

    on='station_id',

    how='left'

)

In [105]:
df2['user_type'].value_counts()

user_type
내국인    2266309
비회원      10250
외국인       1360
Name: count, dtype: int64

In [107]:
df2['is_foreigner'] = (

    df2['user_type']

    ==

    '외국인'

).astype(int)

In [109]:
foreigner_ratio = (

    df2.groupby('station_id')['is_foreigner']

    .mean()

    .rename('foreigner_ratio')

    .reset_index()

)

In [111]:
station_master = station_master.merge(

    foreigner_ratio,

    on='station_id',

    how='left'

)

In [113]:
df2['is_nonmember'] = (

    df2['user_type']

    ==

    '비회원'

).astype(int)

In [115]:
nonmember_ratio = (

    df2.groupby('station_id')['is_nonmember']

    .mean()

    .rename('nonmember_ratio')

    .reset_index()

)

In [117]:
station_master = station_master.merge(

    nonmember_ratio,

    on='station_id',

    how='left'

)

In [119]:
station_master[[

    'station_id',
    'station_name',

    'avg_trip_duration',
    'avg_distance',
    'round_trip_ratio',
    'sprout_bike_ratio',
    'foreigner_ratio',
    'nonmember_ratio'

]].head(20)

,station_id,station_name,avg_trip_duration,avg_distance,round_trip_ratio,sprout_bike_ratio,foreigner_ratio,nonmember_ratio
0,301,경복궁역 7번출구 앞,15.625478,1.487048,0.0,0.091720,0.002548,0.001274
1,302,경복궁역 4번출구 뒤,16.966642,1.635869,0.0,0.058738,0.001450,0.007252
2,303,광화문역 1번출구 앞,15.733066,1.441492,0.0,0.042649,0.000000,0.008530
3,305,종로구청 옆,21.576068,2.109980,0.0,0.034188,0.000855,0.001709
4,307,서울역사박물관 앞,20.488591,1.989540,0.0,0.064430,0.004027,0.004027
5,308,광화문 S타워 앞,14.406222,1.366690,0.0,0.037778,0.000000,0.002667
6,309,광화문역 6번출구 옆 B,18.457213,2.064825,0.0,0.041565,0.000000,0.002445
7,314,국립현대미술관,25.097808,1.918898,0.0,0.030354,0.006745,0.015177
8,316,종각역 1번출구 앞,21.316600,1.730897,0.0,0.028112,0.000669,0.008701
9,326,안국역 5번출구 앞,16.205689,1.626006,0.0,0.043764,0.008753,0.003282


In [121]:
station_master.isnull().sum().sort_values(ascending=False)

lcd_rack_count       1682
qr_ratio             1078
qr_rack_count        1078
nonmember_ratio        37
foreigner_ratio        37
sprout_bike_ratio      37
round_trip_ratio       37
avg_distance           37
avg_trip_duration      37
station_age_days        0
station_id              0
total_rack_count        0
station_name            0
installation_date       0
longitude               0
latitude                0
address                 0
district                0
operation_type          0
dtype: int64

In [123]:
def classify_station_type(row):

    lcd = pd.notnull(row['lcd_rack_count'])

    qr = pd.notnull(row['qr_rack_count'])

    if lcd and qr:
        return 'mixed'

    elif lcd:
        return 'lcd'

    elif qr:
        return 'qr'

    else:
        return 'unknown'

In [125]:
station_master['station_type'] = (

    station_master.apply(
        classify_station_type,
        axis=1
    )

)

In [127]:
station_master['station_type'].value_counts()

station_type
qr       1682
lcd      1078
mixed       6
Name: count, dtype: int64

In [129]:
station_master['lcd_rack_count'] = (
    station_master['lcd_rack_count']
    .fillna(0)
)

station_master['qr_rack_count'] = (
    station_master['qr_rack_count']
    .fillna(0)
)

In [131]:
station_master['total_rack_count'] = (

    station_master['lcd_rack_count']

    +

    station_master['qr_rack_count']

)

In [133]:
station_master['qr_ratio'] = (

    station_master['qr_rack_count']

    /

    station_master['total_rack_count']

)

In [135]:
station_master['qr_ratio'] = (

    station_master['qr_ratio']

    .replace([float('inf'), -float('inf')], 0)

    .fillna(0)

)

In [137]:


print(station_master.shape)

print(station_master.columns)

station_master.head()

(2766, 20)
Index(['station_id', 'station_name', 'district', 'address', 'latitude',
       'longitude', 'installation_date', 'lcd_rack_count', 'qr_rack_count',
       'operation_type', 'total_rack_count', 'qr_ratio', 'station_age_days',
       'avg_trip_duration', 'avg_distance', 'round_trip_ratio',
       'sprout_bike_ratio', 'foreigner_ratio', 'nonmember_ratio',
       'station_type'],
      dtype='object')


,station_id,station_name,district,address,latitude,longitude,installation_date,lcd_rack_count,qr_rack_count,operation_type,total_rack_count,qr_ratio,station_age_days,avg_trip_duration,avg_distance,round_trip_ratio,sprout_bike_ratio,foreigner_ratio,nonmember_ratio,station_type
0,301,경복궁역 7번출구 앞,종로구,서울특별시 종로구 사직로 지하130 경복궁역 7번출구 앞,37.575794,126.971451,2015-10-07 12:03:46,0.0,20.0,QR,20.0,1.0,3883,15.625478,1.487048,0.0,0.091720,0.002548,0.001274,qr
1,302,경복궁역 4번출구 뒤,종로구,서울특별시 종로구 사직로 지하130 경복궁역 4번출구 뒤,37.575947,126.974060,2015-10-07 12:04:22,0.0,12.0,QR,12.0,1.0,3883,16.966642,1.635869,0.0,0.058738,0.001450,0.007252,qr
2,303,광화문역 1번출구 앞,종로구,서울특별시 종로구 세종대로 지하189 세종로공원,37.571770,126.974663,2015-10-07 00:00:00,0.0,8.0,QR,8.0,1.0,3884,15.733066,1.441492,0.0,0.042649,0.000000,0.008530,qr
3,305,종로구청 옆,종로구,서울특별시 종로구 세종로 84-1,37.572559,126.978333,2015-01-07 00:00:00,0.0,16.0,QR,16.0,1.0,4157,21.576068,2.109980,0.0,0.034188,0.000855,0.001709,qr
4,307,서울역사박물관 앞,종로구,서울특별시 종로구 새문안로 55 서울역사박물관 앞,37.570000,126.971100,2015-10-07 12:09:09,0.0,11.0,QR,11.0,1.0,3883,20.488591,1.989540,0.0,0.064430,0.004027,0.004027,qr


In [143]:
station_master.isnull().sum().sort_values(ascending=False)

nonmember_ratio      37
foreigner_ratio      37
sprout_bike_ratio    37
round_trip_ratio     37
avg_distance         37
avg_trip_duration    37
station_id            0
station_name          0
station_age_days      0
qr_ratio              0
total_rack_count      0
operation_type        0
qr_rack_count         0
lcd_rack_count        0
installation_date     0
longitude             0
latitude              0
address               0
district              0
station_type          0
dtype: int64

In [145]:
station_master.describe()

,latitude,longitude,installation_date,lcd_rack_count,qr_rack_count,total_rack_count,qr_ratio,station_age_days,avg_trip_duration,avg_distance,round_trip_ratio,sprout_bike_ratio,foreigner_ratio,nonmember_ratio
count,2766.000000,2766.000000,2766,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2729.000000,2729.000000,2729.0,2729.000000,2729.000000,2729.000000
mean,37.547727,126.991775,2019-05-13 04:51:17.845985024,4.834418,7.254519,12.088937,0.608958,2569.606652,18.861750,2.038937,0.0,0.016282,0.000775,0.005423
min,37.430977,126.798599,2015-01-07 00:00:00,0.000000,0.000000,2.000000,0.000000,550.000000,7.157377,0.882375,0.0,0.000000,0.000000,0.000000
25%,37.505643,126.914110,2017-06-22 10:33:06,0.000000,0.000000,10.000000,0.000000,1967.000000,15.625478,1.615681,0.0,0.004104,0.000000,0.000929
50%,37.545483,127.005554,2018-12-14 00:00:00,0.000000,8.000000,10.000000,1.000000,2720.000000,18.239870,1.892200,0.0,0.009304,0.000000,0.003224
75%,37.577515,127.064451,2021-01-05 00:00:00,10.000000,10.000000,15.000000,1.000000,3259.000000,21.248933,2.269310,0.0,0.020704,0.000000,0.006494
max,37.691013,127.180756,2024-11-22 00:00:00,40.000000,62.000000,62.000000,1.000000,4157.000000,55.895706,6.935396,0.0,0.239175,0.062069,0.189691
std,0.052404,0.092856,NaN,6.664477,7.453407,5.528035,0.487611,806.836431,5.085706,0.670132,0.0,0.020705,0.002963,0.009253


In [147]:
Time['station_id'] = (

    Time['대여소번호']

    .astype(str)

    .str.replace('.0', '', regex=False)

    .str.strip()

)

In [148]:
Time['대여일자'] = pd.to_datetime(
    Time['대여일자']
)

In [149]:
Time['weekday'] = (
    Time['대여일자'].dt.weekday
)

In [153]:
Time['hour'] = (
    Time['대여시간']
    .astype(int)
)

In [155]:
Time['is_weekend'] = (

    Time['weekday']

    .isin([5,6])

).astype(int)

In [157]:
missing_station_ids = set(station_master['station_id']) - set(df2['station_id'])

print(len(missing_station_ids))
print(sorted(list(missing_station_ids))[:50])

37
['1026', '1035', '1080', '1128', '2076', '211', '2113', '212', '2135', '2275', '2282', '2298', '2316', '2322', '2355', '2421', '2527', '253', '2914', '3305', '3603', '3605', '3621', '3655', '3691', '3779', '3787', '4519', '4590', '4709', '4901', '515', '5854', '5855', '764', '790', '835']


In [159]:
station_master[
    station_master['station_id'].isin(missing_station_ids)
][[
    'station_id',
    'station_name',
    'district'
]]

,station_id,station_name,district
95,4709,천지인 오피스텔 앞,종로구
211,835,남영역 건너편,용산구
391,515,광양중학교 앞,광진구
910,2914,서울과학기술대학교(미래관),노원구
1316,764,목동청소년수련관,양천구
1337,790,화곡고가 사거리,양천구
1354,4519,양원보도육교,양천구
1396,1128,화곡역 6번출구,강서구
1508,3779,유광사여성병원,강서구
1516,3787,가양나들목,강서구


In [161]:
missing_ids = [
    4709,835,515,2914,764,790,4519,1128,3779,3787,
    211,212,253,4590,5854,5855,2076,2113,2135,3305,
    2275,2282,2298,2527,2316,2322,2355,2421,3603,
    3605,3621,4901,1026,1035,1080,3655,3691
]

df2[df2['station_id'].isin(missing_ids)]

,bike_number,rental_datetime,rental_station_id,rental_station_name,대여거치대,return_datetime,return_station_id,return_station_name,return_rack,usage_time,...,user_type,대여대여소ID,반납대여소ID,bike_type,station_id,distance_km,is_round_trip,is_sprout_bike,is_foreigner,is_nonmember


In [336]:
weekend_usage = (

    Time.loc[Time['is_weekend']==1]

    .groupby('station_id')['이용건수']

    .sum()

)

In [338]:
total_usage = (

    Time.groupby('station_id')['이용건수']

    .sum()

)

In [340]:
weekend_ratio = (

    weekend_usage

    /

    total_usage

).rename('weekend_ratio').reset_index()

In [342]:
station_master = station_master.merge(

    weekend_ratio,

    on='station_id',

    how='left'

)

In [344]:
# RUSH HOUR RATIO


rush_hours = [7,8,9,17,18,19]

Time['is_rush_hour'] = (
    Time['hour'].isin(rush_hours)
).astype(int)

rush_usage = (

    Time.loc[
        Time['is_rush_hour'] == 1
    ]

    .groupby('station_id')['이용건수']

    .sum()

)

total_usage = (

    Time.groupby('station_id')['이용건수']

    .sum()

)

rush_hour_ratio = (

    rush_usage
    /
    total_usage

).rename('rush_hour_ratio').reset_index()


station_master = station_master.merge(

    rush_hour_ratio,

    on='station_id',

    how='left'

)

In [346]:
# LATE NIGHT RATIO


late_hours = [0,1,2,3,4,5]

Time['is_late_night'] = (
    Time['hour'].isin(late_hours)
).astype(int)

late_usage = (

    Time.loc[
        Time['is_late_night'] == 1
    ]

    .groupby('station_id')['이용건수']

    .sum()

)

total_usage = (

    Time.groupby('station_id')['이용건수']

    .sum()

)

late_night_ratio = (

    late_usage
    /
    total_usage

).rename('late_night_ratio').reset_index()



station_master = station_master.merge(

    late_night_ratio,

    on='station_id',

    how='left'

)

In [348]:
# HOURLY ENTROPY


from scipy.stats import entropy

hourly_distribution = (

    Time.pivot_table(

        index='station_id',

        columns='hour',

        values='이용건수',

        aggfunc='sum',

        fill_value=0

    )

)

hourly_entropy = (

    hourly_distribution.apply(

        lambda x: entropy(x / x.sum()),

        axis=1

    )

    .rename('hourly_entropy')

    .reset_index()

)


station_master = station_master.merge(

    hourly_entropy,

    on='station_id',

    how='left'

)

In [350]:

# PEAK CONCENTRATION

peak_concentration = (

    (

        hourly_distribution.max(axis=1)

        /

        hourly_distribution.sum(axis=1)

    )

    .rename('peak_concentration')

    .reset_index()

)



station_master = station_master.merge(

    peak_concentration,

    on='station_id',

    how='left'

)

In [352]:
#TEMPORAL STABILITY

daily_station_usage = (

    Time.groupby([

        'station_id',
        '대여일자'

    ])['이용건수']

    .sum()

    .reset_index()

)

temporal_stability = (

    daily_station_usage.groupby('station_id')['이용건수']

    .agg(['mean', 'std'])

)

temporal_stability['temporal_stability'] = (

    1

    -

    (

        temporal_stability['std']

        /

        temporal_stability['mean']

    )

)

temporal_stability = (

    temporal_stability['temporal_stability']

    .reset_index()

)


station_master = station_master.merge(

    temporal_stability,

    on='station_id',

    how='left'

)

In [356]:

# WEEKEND RATIO

Time['is_weekend'] = (

    Time['weekday']

    .isin([5,6])

).astype(int)

weekend_usage = (

    Time.loc[
        Time['is_weekend'] == 1
    ]

    .groupby('station_id')['이용건수']

    .sum()

)

total_usage = (

    Time.groupby('station_id')['이용건수']

    .sum()

)

weekend_ratio = (

    weekend_usage
    /
    total_usage

).rename('weekend_ratio').reset_index()


station_master = station_master.merge(

    weekend_ratio,

    on='station_id',

    how='left'

)

In [358]:

# TEMPORAL FEATURE CHECK

station_master[[

    'station_id',
    'station_name',

    'weekend_ratio',
    'rush_hour_ratio',
    'late_night_ratio',

    'hourly_entropy',
    'peak_concentration',
    'temporal_stability'

]].head(20)

,station_id,station_name,weekend_ratio,rush_hour_ratio,late_night_ratio,hourly_entropy,peak_concentration,temporal_stability
0,301,경복궁역 7번출구 앞,0.169427,0.527389,0.048408,2.801663,0.200000,0.597742
1,302,경복궁역 4번출구 뒤,0.171139,0.469181,0.052212,2.910802,0.148658,0.628937
2,303,광화문역 1번출구 앞,0.115906,0.473658,0.033116,2.826138,0.121927,0.552688
3,305,종로구청 옆,0.132479,0.435043,0.026496,2.830128,0.121368,0.500319
4,307,서울역사박물관 앞,0.127517,0.382550,0.053691,2.877141,0.132886,0.512546
5,308,광화문 S타워 앞,0.081887,0.483311,0.020917,2.651616,0.173120,0.394739
6,309,광화문역 6번출구 옆 B,0.090465,0.528117,0.051345,2.712614,0.246944,0.368587
7,314,국립현대미술관,0.278246,0.370995,0.005059,2.608353,0.155143,0.738011
8,316,종각역 1번출구 앞,0.121821,0.485944,0.036814,2.829858,0.121821,0.498149
9,326,안국역 5번출구 앞,0.141138,0.480306,0.048140,2.826755,0.131291,0.561447


In [360]:
Month['station_id'] = (

    Month['대여소번호']

    .astype(str)

    .str.replace('.0', '', regex=False)

    .str.strip()

)

In [362]:
Month['is_male'] = (

    Month['성별']

    ==

    'M'

).astype(int)

In [364]:
male_usage = (

    Month.loc[
        Month['is_male'] == 1
    ]

    .groupby('station_id')['이용건수']

    .sum()

)

In [366]:
total_usage = (

    Month.groupby('station_id')['이용건수']

    .sum()

)

In [368]:
male_ratio = (

    male_usage

    /

    total_usage

).rename('male_ratio').reset_index()

In [370]:
station_master = station_master.merge(

    male_ratio,

    on='station_id',

    how='left'

)

In [372]:
from scipy.stats import entropy

age_distribution = (

    Month.pivot_table(

        index='station_id',

        columns='연령대코드',

        values='이용건수',

        aggfunc='sum',

        fill_value=0

    )

)

In [374]:
age_entropy = (

    age_distribution.apply(

        lambda x: entropy(x / x.sum()),

        axis=1

    )

    .rename('age_entropy')

    .reset_index()

)

In [376]:
station_master = station_master.merge(

    age_entropy,

    on='station_id',

    how='left'

)

In [378]:
print(Month['연령대코드'].value_counts())

연령대코드
기타       101482
20대       97123
30대       95628
40대       91184
50대       82530
~10대      66756
60대       59331
70대이상     25630
Name: count, dtype: int64


In [384]:
Month['is_other_age'] = (

    Month['연령대코드']

    ==

    '기타'

).astype(int)

In [386]:
other_usage = (

    Month.loc[
        Month['is_other_age'] == 1
    ]

    .groupby('station_id')['이용건수']

    .sum()

)

In [388]:
total_usage = (

    Month.groupby('station_id')['이용건수']

    .sum()

)

In [390]:
other_age_ratio = (

    other_usage

    /

    total_usage

).rename('other_age_ratio').reset_index()

In [392]:
station_master = station_master.merge(

    other_age_ratio,

    on='station_id',

    how='left'

)

In [394]:
Month_entropy = (

    Month.loc[
        Month['연령대코드'] != '기타'
    ]

    .copy()

)

In [396]:
# YOUTH RATIO
# ~10s+ 20s


youth_codes = [

    '~10대',
    '20대'

]


youth_usage = (

    Month.loc[
        Month['연령대코드'].isin(youth_codes)
    ]

    .groupby('station_id')['이용건수']

    .sum()

)


total_usage = (

    Month.groupby('station_id')['이용건수']

    .sum()

)



youth_ratio = (

    youth_usage

    /

    total_usage

).rename('youth_ratio').reset_index()



station_master = station_master.merge(

    youth_ratio,

    on='station_id',

    how='left'

)

In [398]:
station_master[[

    'station_id',
    'station_name',
    'youth_ratio'

]].head(20)

,station_id,station_name,youth_ratio
0,301,경복궁역 7번출구 앞,0.256824
1,302,경복궁역 4번출구 뒤,0.290538
2,303,광화문역 1번출구 앞,0.188571
3,305,종로구청 옆,0.198063
4,307,서울역사박물관 앞,0.240349
5,308,광화문 S타워 앞,0.183967
6,309,광화문역 6번출구 옆 B,0.191529
7,314,국립현대미술관,0.280922
8,316,종각역 1번출구 앞,0.236932
9,326,안국역 5번출구 앞,0.254837


In [400]:
# SENIOR RATIO


senior_codes = [

    '60대',
    '70대이상'

]


senior_usage = (

    Month.loc[
        Month['연령대코드'].isin(senior_codes)
    ]

    .groupby('station_id')['이용건수']

    .sum()

)


total_usage = (

    Month.groupby('station_id')['이용건수']

    .sum()

)


senior_ratio = (

    senior_usage

    /

    total_usage

).rename('senior_ratio').reset_index()



station_master = station_master.merge(

    senior_ratio,

    on='station_id',

    how='left'

)

In [402]:
# MIDDLE AGE RATIO


middle_codes = [

    '30대',
    '40대',
    '50대'

]


middle_usage = (

    Month.loc[
        Month['연령대코드'].isin(middle_codes)
    ]

    .groupby('station_id')['이용건수']

    .sum()

)


total_usage = (

    Month.groupby('station_id')['이용건수']

    .sum()

)


middle_age_ratio = (

    middle_usage

    /

    total_usage

).rename('middle_age_ratio').reset_index()


station_master = station_master.merge(

    middle_age_ratio,

    on='station_id',

    how='left'

)

In [404]:
# AGE ENTROPY
# same as age diversity
from scipy.stats import entropy

# entropy calciation , I extracet 기타 value
Month_entropy = (

    Month.loc[
        Month['연령대코드'] != '기타'
    ]

    .copy()

)



age_distribution = (

    Month_entropy.pivot_table(

        index='station_id',

        columns='연령대코드',

        values='이용건수',

        aggfunc='sum',

        fill_value=0

    )

)


age_entropy = (

    age_distribution.apply(

        lambda x: entropy(x / x.sum()),

        axis=1

    )

    .rename('age_entropy')

    .reset_index()

)



station_master = station_master.merge(

    age_entropy,

    on='station_id',

    how='left'

)

In [416]:
print(station_master.columns.tolist())

['station_id', 'station_name', 'district', 'address', 'latitude', 'longitude', 'installation_date', 'lcd_rack_count', 'qr_rack_count', 'operation_type', 'total_rack_count', 'qr_ratio', 'station_age_days', 'avg_trip_duration', 'avg_distance', 'round_trip_ratio', 'sprout_bike_ratio', 'foreigner_ratio', 'nonmember_ratio', 'station_type', 'rush_hour_ratio', 'late_night_ratio', 'hourly_entropy', 'peak_concentration', 'temporal_stability', 'weekend_ratio', 'male_ratio', 'other_age_ratio', 'youth_ratio', 'senior_ratio', 'middle_age_ratio', 'age_entropy']


In [414]:
station_master = station_master.drop(columns=[

    'weekend_ratio_x',
    'weekend_ratio_y'

])

In [418]:


station_master[[

    'station_id',
    'station_name',

    'male_ratio',

    'youth_ratio',
    'middle_age_ratio',
    'senior_ratio',
    'other_age_ratio',

    'age_entropy'

]].head(20)

,station_id,station_name,male_ratio,youth_ratio,middle_age_ratio,senior_ratio,other_age_ratio,age_entropy
0,301,경복궁역 7번출구 앞,0.505585,0.256824,0.622050,0.050121,0.071005,1.571667
1,302,경복궁역 4번출구 뒤,0.484340,0.290538,0.573147,0.053456,0.082860,1.549042
2,303,광화문역 1번출구 앞,0.494371,0.188571,0.663649,0.075757,0.072023,1.574669
3,305,종로구청 옆,0.532669,0.198063,0.693084,0.055518,0.053335,1.565585
4,307,서울역사박물관 앞,0.555322,0.240349,0.646261,0.050719,0.062672,1.551401
5,308,광화문 S타워 앞,0.562411,0.183967,0.732913,0.034998,0.048122,1.450958
6,309,광화문역 6번출구 옆 B,0.572539,0.191529,0.684060,0.066912,0.057499,1.601896
7,314,국립현대미술관,0.446324,0.280922,0.591453,0.047987,0.079638,1.563958
8,316,종각역 1번출구 앞,0.506841,0.236932,0.630684,0.054409,0.077975,1.537235
9,326,안국역 5번출구 앞,0.515627,0.254837,0.625987,0.049800,0.069376,1.581490


In [420]:
df_december['station_id'] = (

    df_december['시작_대여소_ID']

    .astype(str)

    .str.replace('ST-', '', regex=False)

    .str.replace('.0', '', regex=False)

    .str.strip()

)

In [424]:
df_december['destination_id'] = (

    df_december['종료_대여소_ID']

    .astype(str)

    .str.replace('ST-', '', regex=False)

    .str.replace('.0', '', regex=False)

    .str.strip()

)

In [426]:
from scipy.stats import entropy

destination_distribution = (

    df_december.pivot_table(

        index='station_id',

        columns='destination_id',

        values='전체_건수',

        aggfunc='sum',

        fill_value=0

    )

)

In [428]:
destination_entropy = (

    destination_distribution.apply(

        lambda x: entropy(x / x.sum()),

        axis=1

    )

    .rename('destination_entropy')

    .reset_index()

)

In [430]:
station_master = station_master.merge(

    destination_entropy,

    on='station_id',

    how='left'

)

In [440]:
# =========================================================
# OUTFLOW
# =========================================================

outflow = (

    df_december.groupby('station_id')['전체_건수']

    .sum()

)

# =========================================================
# INFLOW
# =========================================================

inflow = (

    df_december.groupby('destination_id')['전체_건수']

    .sum()

)

# =========================================================
# ALIGN INDEX NAME
# =========================================================

inflow.index.name = 'station_id'

# =========================================================
# RATIO
# =========================================================

inflow_outflow_ratio = (

    inflow

    /

    outflow

).rename('inflow_outflow_ratio')

# =========================================================
# DATAFRAME 변환
# =========================================================

inflow_outflow_ratio = (

    inflow_outflow_ratio

    .reset_index()

)

In [442]:
print(inflow_outflow_ratio.head())

print(inflow_outflow_ratio.columns)

  station_id  inflow_outflow_ratio
0         10              1.074919
1       1000              1.016026
2       1002              1.042925
3       1004              0.695238
4       1005              0.996648
Index(['station_id', 'inflow_outflow_ratio'], dtype='object')


In [444]:
station_master = station_master.merge(

    inflow_outflow_ratio,

    on='station_id',

    how='left'

)

In [446]:
import networkx as nx

In [448]:
edge_weights = (

    df_december.groupby([

        'station_id',
        'destination_id'

    ])['전체_건수']

    .sum()

    .reset_index()

)

In [450]:
G = nx.from_pandas_edgelist(

    edge_weights,

    source='station_id',

    target='destination_id',

    edge_attr='전체_건수',

    create_using=nx.DiGraph()

)

In [452]:
degree_centrality = pd.Series(

    nx.degree_centrality(G),

    name='degree_centrality'

).reset_index()

degree_centrality.columns = [

    'station_id',
    'degree_centrality'

]

In [454]:
station_master = station_master.merge(

    degree_centrality,

    on='station_id',

    how='left'

)

In [458]:
closeness_centrality = pd.Series(

    nx.closeness_centrality(G),

    name='closeness_centrality'

).reset_index()

closeness_centrality.columns = [

    'station_id',
    'closeness_centrality'

]

In [460]:
station_master = station_master.merge(

    closeness_centrality,

    on='station_id',

    how='left'

)

In [464]:
betweenness_centrality = pd.Series(

    nx.betweenness_centrality(

        G,

        weight='전체_건수'

    ),

    name='betweenness_centrality'

).reset_index()

betweenness_centrality.columns = [

    'station_id',
    'betweenness_centrality'

]

KeyboardInterrupt: 

In [ ]:
# it do not work well. I didn't use this column

In [466]:
station_master[[

    'station_id',
    'station_name',

    'destination_entropy',
    'inflow_outflow_ratio',

    'degree_centrality',
    'closeness_centrality',
    

]].head(20)

,station_id,station_name,destination_entropy,inflow_outflow_ratio,degree_centrality,closeness_centrality
0,301,경복궁역 7번출구 앞,NaN,NaN,NaN,NaN
1,302,경복궁역 4번출구 뒤,4.140862,1.053792,0.099198,0.404017
2,303,광화문역 1번출구 앞,3.877326,0.992530,0.081692,0.380265
3,305,종로구청 옆,NaN,NaN,NaN,NaN
4,307,서울역사박물관 앞,3.634888,1.029791,0.113786,0.455242
5,308,광화문 S타워 앞,4.416383,0.909732,0.107221,0.395777
6,309,광화문역 6번출구 옆 B,3.897997,1.005677,0.105033,0.406297
7,314,국립현대미술관,NaN,NaN,NaN,NaN
8,316,종각역 1번출구 앞,3.567246,0.997812,0.047411,0.338238
9,326,안국역 5번출구 앞,3.553618,1.019820,0.063457,0.399009


In [468]:
network_cols = [

    'destination_entropy',
    'inflow_outflow_ratio',

    'degree_centrality',
    'closeness_centrality'

]

station_master[network_cols] = (

    station_master[network_cols]

    .replace([float('inf'), -float('inf')], 0)

    .fillna(0)

)

In [470]:
station_master[network_cols].isnull().sum()

destination_entropy     0
inflow_outflow_ratio    0
degree_centrality       0
closeness_centrality    0
dtype: int64

In [472]:

# FOREIGNER ECOLOGY FEATURE ENGINEERING

Forienr['station_id'] = (

    Forienr['대여소명']

    .str.extract(r'^(\d+)')

)



Forienr['일시'] = pd.to_datetime(
    Forienr['일시']
)


Forienr['total_foreigner_flow'] = (

    Forienr['대여건수']

    +

    Forienr['반납건수']

)


# FOREIGNER ACTIVITY

foreigner_activity = (

    Forienr.groupby('station_id')[

        'total_foreigner_flow'

    ]

    .sum()

    .rename('foreigner_activity')

    .reset_index()

)

station_master = station_master.merge(

    foreigner_activity,

    on='station_id',

    how='left'

)

# FEATURE 2
# FOREIGNER BALANCE

station_flow = (

    Forienr.groupby('station_id')[

        ['대여건수', '반납건수']

    ]

    .sum()

)

station_flow['foreigner_balance'] = (

    station_flow['반납건수']

    /

    station_flow['대여건수']

)

foreigner_balance = (

    station_flow['foreigner_balance']

    .replace([float('inf'), -float('inf')], 0)

    .fillna(0)

    .reset_index()

)

station_master = station_master.merge(

    foreigner_balance,

    on='station_id',

    how='left'

)

# FEATURE 3
# FOREIGNER STABILITY

daily_foreigner = (

    Forienr.groupby([

        'station_id',
        '일시'

    ])['total_foreigner_flow']

    .sum()

    .reset_index()

)

foreigner_stability = (

    daily_foreigner.groupby('station_id')[

        'total_foreigner_flow'

    ]

    .agg(['mean', 'std'])

)

foreigner_stability['foreigner_stability'] = (

    1

    -

    (

        foreigner_stability['std']

        /

        foreigner_stability['mean']

    )

)

foreigner_stability = (

    foreigner_stability['foreigner_stability']

    .replace([float('inf'), -float('inf')], 0)

    .fillna(0)

    .reset_index()

)

station_master = station_master.merge(

    foreigner_stability,

    on='station_id',

    how='left'

)


# FEATURE 4
# FOREIGNER DAILY MEAN

foreigner_daily_mean = (

    daily_foreigner.groupby('station_id')[

        'total_foreigner_flow'

    ]

    .mean()

    .rename('foreigner_daily_mean')

    .reset_index()

)

station_master = station_master.merge(

    foreigner_daily_mean,

    on='station_id',

    how='left'

)


foreigner_cols = [

    'foreigner_activity',
    'foreigner_balance',
    'foreigner_stability',
    'foreigner_daily_mean'

]

station_master[foreigner_cols] = (

    station_master[foreigner_cols]

    .replace([float('inf'), -float('inf')], 0)

    .fillna(0)

)

=========

station_master[[

    'station_id',
    'station_name',

    'foreigner_activity',
    'foreigner_balance',
    'foreigner_stability',
    'foreigner_daily_mean'

]].head(20)

,station_id,station_name,foreigner_activity,foreigner_balance,foreigner_stability,foreigner_daily_mean
0,301,경복궁역 7번출구 앞,61.0,0.196078,0.420806,1.967742
1,302,경복궁역 4번출구 뒤,377.0,0.984211,0.182805,4.010638
2,303,광화문역 1번출구 앞,59.0,0.552632,0.358085,2.565217
3,305,종로구청 옆,66.0,1.062500,0.408887,1.650000
4,307,서울역사박물관 앞,38.0,1.235294,0.457997,1.809524
5,308,광화문 S타워 앞,26.0,0.529412,0.637835,1.368421
6,309,광화문역 6번출구 옆 B,18.0,1.000000,0.587989,1.636364
7,314,국립현대미술관,154.0,0.811765,0.305815,2.298507
8,316,종각역 1번출구 앞,61.0,0.794118,0.359631,1.794118
9,326,안국역 5번출구 앞,162.0,0.951807,0.251850,2.531250


In [478]:
#Saving

station_master.to_csv(

    '/Users/choejeonghun/Downloads/station_ecology_master.csv',

    index=False,

    encoding='utf-8-sig'

)

print('Saved: station_ecology_master.csv')

Saved: station_ecology_master.csv


In [ ]:



file_path = "/Users/choejeonghun/Downloads/Fianl Project_CDS/공공자전거 대여소 정보(25.6월 기준).xlsx"

raw = pd.read_excel(
    file_path,
    header=None
)


df = raw.iloc[5:].copy()

# 컬럼명 직접 지정
df.columns = [
    "station_id",
    "station_name",
    "district",
    "address",
    "latitude",
    "longitude",
    "install_date",
    "install_LCD",
    "install_QR",
    "operation_type"
]



df["latitude"] = pd.to_numeric(
    df["latitude"],
    errors="coerce"
)

df["longitude"] = pd.to_numeric(
    df["longitude"],
    errors="coerce"
)

# 결측 제거
df = df.dropna(
    subset=["district", "station_name", "latitude", "longitude"]
)

print(df.head())



In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

adjacency = {

    "강남구": ["서초구", "송파구", "용산구", "성동구", "광진구"],
    "강동구": ["송파구", "광진구"],
    "강북구": ["도봉구", "노원구", "성북구", "종로구"],
    "강서구": ["양천구", "영등포구", "마포구"],
    "관악구": ["동작구", "금천구", "서초구"],
    "광진구": ["성동구", "동대문구", "중랑구",
             "송파구", "강동구", "강남구"],
    "구로구": ["양천구", "영등포구", "금천구", "관악구"],
    "금천구": ["구로구", "관악구", "동작구", "영등포구"],
    "노원구": ["도봉구", "강북구", "성북구", "중랑구"],
    "도봉구": ["노원구", "강북구"],
    "동대문구": ["종로구", "성북구", "중랑구",
               "광진구", "성동구"],
    "동작구": ["영등포구", "관악구", "서초구",
             "용산구", "금천구"],
    "마포구": ["은평구", "서대문구", "용산구",
             "강서구", "영등포구"],
    "서대문구": ["은평구", "종로구", "마포구", "중구"],
    "서초구": ["동작구", "관악구", "강남구", "용산구"],
    "성동구": ["중구", "용산구", "동대문구",
             "광진구", "강남구"],
    "성북구": ["강북구", "노원구", "중랑구",
             "동대문구", "종로구"],
    "송파구": ["강남구", "강동구", "광진구"],
    "양천구": ["강서구", "구로구", "영등포구"],
    "영등포구": ["강서구", "양천구", "구로구",
               "금천구", "동작구", "마포구", "용산구"],
    "용산구": ["마포구", "중구", "성동구",
             "영등포구", "동작구",
             "서초구", "강남구"],
    "은평구": ["마포구", "서대문구", "종로구"],
    "종로구": ["은평구", "서대문구", "중구",
             "성북구", "동대문구", "강북구"],
    "중구": ["종로구", "용산구", "성동구", "서대문구"],
    "중랑구": ["노원구", "성북구", "동대문구", "광진구"]
}


G = nx.Graph()

for district, neighbors in adjacency.items():

    for neighbor in neighbors:

        G.add_edge(district, neighbor)



pos = {

    # -----------------------------
    # 도심권
    # -----------------------------
    "종로구": (0, 2),
    "중구": (0.7, 1.5),
    "용산구": (0.5, 0.7),

    # -----------------------------
    # 동북권
    # -----------------------------
    "도봉구": (2.5, 5),
    "노원구": (3.5, 4.5),
    "강북구": (1.8, 4.2),
    "성북구": (2.2, 3.2),
    "동대문구": (3.2, 2.5),
    "중랑구": (4.2, 3),
    "성동구": (3.2, 1.4),
    "광진구": (4.5, 1.5),

    # -----------------------------
    # 서북권
    # -----------------------------
    "은평구": (-2.2, 3.2),
    "서대문구": (-1.2, 2.2),
    "마포구": (-1.5, 1),

    # -----------------------------
    # 서남권
    # -----------------------------
    "강서구": (-4, 0.5),
    "양천구": (-3, -0.2),
    "영등포구": (-1.7, -0.3),
    "구로구": (-3, -1.5),
    "금천구": (-2.2, -2.5),
    "동작구": (-0.5, -1.2),
    "관악구": (-1.2, -2.4),

    # -----------------------------
    # 동남권
    # -----------------------------
    "서초구": (1.5, -1),
    "강남구": (3, -0.7),
    "송파구": (5, -0.5),
    "강동구": (6, 0.5)
}

# =====================================================
# 권역별 색상
# =====================================================

region_colors = {

    # 도심권
    "종로구": "red",
    "중구": "red",
    "용산구": "red",

    # 동북권
    "성동구": "skyblue",
    "광진구": "skyblue",
    "동대문구": "skyblue",
    "중랑구": "skyblue",
    "성북구": "skyblue",
    "강북구": "skyblue",
    "도봉구": "skyblue",
    "노원구": "skyblue",

    # 서북권
    "은평구": "orange",
    "서대문구": "orange",
    "마포구": "orange",

    # 서남권
    "양천구": "green",
    "강서구": "green",
    "구로구": "green",
    "금천구": "green",
    "영등포구": "green",
    "동작구": "green",
    "관악구": "green",

    # 동남권
    "서초구": "purple",
    "강남구": "purple",
    "송파구": "purple",
    "강동구": "purple"
}

node_colors = [
    region_colors[node]
    for node in G.nodes()
]

#visualization
plt.figure(figsize=(18, 15))

# edge
nx.draw_networkx_edges(
    G,
    pos,
    width=2,
    alpha=0.7
)

# node
nx.draw_networkx_nodes(
    G,
    pos,
    node_size=3200,
    node_color=node_colors,
    alpha=0.95
)

# label
nx.draw_networkx_labels(
    G,
    pos,
    font_size=10,
    font_family="AppleGothic",
    font_color="black"
)


plt.plot(
    [-5, 7],
    [0.2, 0.2],
    linewidth=8,
    alpha=0.25
)

plt.text(
    6.5,
    0.4,
    "한강",
    fontsize=12
)


plt.title(
    "Seoul Geographic District Network",
    fontsize=22
)

plt.axis("off")

plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import platform

# 1. OS별 한글 폰트 설정 (Windows / Mac)
os_name = platform.system()
if os_name == "Windows": font_family = "Malgun Gothic"
elif os_name == "Darwin": font_family = "AppleGothic"
else: font_family = "DejaVu Sans"
plt.rcParams['font.family'] = font_family
plt.rcParams['axes.unicode_minus'] = False

# 2. 서울 자치구 인접 데이터 및 그래프 생성
adjacency = {
    "종로구": ["은평구", "서대문구", "중구", "성북구", "동대문구", "강북구"],
    "중구": ["종로구", "용산구", "성동구", "서대문구", "마포구"],
    "용산구": ["마포구", "중구", "성동구", "영등포구", "동작구", "서초구", "강남구"],
    "성동구": ["중구", "용산구", "동대문구", "광진구", "강남구"],
    "광진구": ["성동구", "동대문구", "중랑구", "송파구", "강동구", "강남구"],
    "동대문구": ["종로구", "성북구", "중랑구", "광진구", "성동구"],
    "중랑구": ["노원구", "성북구", "동대문구", "광진구"],
    "성북구": ["강북구", "노원구", "중랑구", "동대문구", "종로구"],
    "강북구": ["도봉구", "노원구", "성북구", "종로구"],
    "도봉구": ["노원구", "강북구"],
    "노원구": ["도봉구", "강북구", "성북구", "중랑구"],
    "은평구": ["마포구", "서대문구", "종로구"],
    "서대문구": ["은평구", "종로구", "마포구", "중구"],
    "마포구": ["은평구", "서대문구", "용산구", "강서구", "영등포구", "중구"],
    "양천구": ["강서구", "구로구", "영등포구"],
    "강서구": ["양천구", "영등포구", "마포구"],
    "구로구": ["양천구", "영등포구", "금천구"],
    "금천구": ["구로구", "관악구", "영등포구"],
    "영등포구": ["강서구", "양천구", "구로구", "금천구", "동작구", "마포구", "용산구"],
    "동작구": ["영등포구", "관악구", "서초구", "용산구"],
    "관악구": ["동작구", "금천구", "서초구"],
    "서초구": ["동작구", "관악구", "강남구", "용산구"],
    "강남구": ["서초구", "송파구", "용산구", "성동구", "광진구"],
    "송파구": ["강남구", "강동구", "광진구"],
    "강동구": ["송파구", "광진구"]
}

G = nx.Graph()
for district, neighbors in adjacency.items():
    for neighbor in neighbors: G.add_edge(district, neighbor)

# 3. 콘솔창(터미널)에서 초기값 물어보기
print("="*40)
print(" 🗺️  서울시 자치구 최단 경로 검색기")
print("="*40)

all_districts = list(G.nodes)

while True:
    start_node = input("▶ 출발 자치구를 입력하세요 (예: 강서구): ").strip()
    if start_node in all_districts: break
    print("❌ 올바른 자치구 이름이 아닙니다. 다시 입력해주세요.")

while True:
    end_node = input("▶ 목적 자치구를 입력하세요 (예: 강동구): ").strip()
    if end_node in all_districts: break
    print("❌ 올바른 자치구 이름이 아닙니다. 다시 입력해주세요.")

# 최단 경로 계산
shortest_path = nx.shortest_path(G, source=start_node, target=end_node)
path_edges = list(zip(shortest_path[:-1], shortest_path[1:]))

print("\n[계산 완료]")
print(f"📍 {start_node} → {end_node} 최단 경로: {' -> '.join(shortest_path)}\n")

# 4. 시각화 세팅
pos = {
    "종로구": (0, 2), "중구": (0.7, 1.5), "용산구": (0.5, 0.7),
    "도봉구": (2.5, 5), "노원구": (3.5, 4.5), "강북구": (1.8, 4.2),
    "성북구": (2.2, 3.2), "동대문구": (3.2, 2.5), "중랑구": (4.2, 3),
    "성동구": (3.2, 1.4), "광진구": (4.5, 1.5), "은평구": (-2.2, 3.2),
    "서대문구": (-1.2, 2.2), "마포구": (-1.5, 1), "강서구": (-4, 0.5),
    "양천구": (-3, -0.2), "영등포구": (-1.7, -0.3), "구로구": (-3, -1.5),
    "금천구": (-2.2, -2.5), "동작구": (-0.5, -1.2), "관악구": (-1.2, -2.4),
    "서초구": (1.5, -1), "강남구": (3, -0.7), "송파구": (5, -0.5), "강동구": (6, 0.5)
}
region_colors = {
    "종로구": "red", "중구": "red", "용산구": "red",
    "성동구": "skyblue", "광진구": "skyblue", "동대문구": "skyblue", "중랑구": "skyblue",
    "성북구": "skyblue", "강북구": "skyblue", "도봉구": "skyblue", "노원구": "skyblue",
    "은평구": "orange", "서대문구": "orange", "마포구": "orange",
    "양천구": "green", "강서구": "green", "구로구": "green", "금천구": "green",
    "영등포구": "green", "동작구": "green", "관악구": "green",
    "서초구": "purple", "강남구": "purple", "송파구": "purple", "강동구": "purple"
}
node_colors = [region_colors[node] for node in G.nodes()]

plt.figure(figsize=(18, 15))

# 기본 그래프 컴포넌트
nx.draw_networkx_edges(G, pos, width=1.5, alpha=0.3, edge_color="gray")
nx.draw_networkx_nodes(G, pos, node_size=3200, node_color=node_colors, alpha=0.85)

# 최단 경로 하이라이트
nx.draw_networkx_edges(G, pos, edgelist=path_edges, width=6, edge_color="crimson", alpha=0.9)
nx.draw_networkx_nodes(G, pos, nodelist=shortest_path, node_size=3200,
                       node_color=[region_colors[n] for n in shortest_path],
                       edgecolors="crimson", linewidths=4, alpha=1.0)

nx.draw_networkx_labels(G, pos, font_size=10, font_family=font_family)
plt.title(f"Seoul Shortest Path: {start_node} → {end_node}", fontsize=22)
plt.axis("off")
plt.show()


In [ ]:


G1 = nx.Graph()

for _, row in df.iterrows():

    district = row["district"]
    station = row["station_name"]

    G1.add_node(
        district,
        node_type="district"
    )

    G1.add_node(
        station,
        node_type="station"
    )

    G1.add_edge(
        district,
        station
    )

# ---------------- 시각화 ----------------

plt.figure(figsize=(22,18))

pos = nx.spring_layout(
    G1,
    seed=42,
    k=0.12
)

district_nodes = [
    n for n, d in G1.nodes(data=True)
    if d["node_type"] == "district"
]

station_nodes = [
    n for n, d in G1.nodes(data=True)
    if d["node_type"] == "station"
]

nx.draw_networkx_nodes(
    G1,
    pos,
    nodelist=district_nodes,
    node_size=2500
)

nx.draw_networkx_nodes(
    G1,
    pos,
    nodelist=station_nodes,
    node_size=20,
    alpha=0.7
)

nx.draw_networkx_edges(
    G1,
    pos,
    alpha=0.2
)

nx.draw_networkx_labels(
    G1,
    pos,
    labels={n:n for n in district_nodes},
    font_size=10
)

plt.title("District - Station Network")
plt.axis("off")
plt.show()

